In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# 1. Carregamento
df = pd.read_csv('/dataset_final_202605222020.csv')
df['data_referencia'] = pd.to_datetime(df['data_referencia'], format='%d/%m/%Y')
df.set_index('data_referencia', inplace=True)
df.sort_index(inplace=True)
df = df.asfreq('D')

# Treatment of missing values
df['preco_brent'] = df['preco_brent'].ffill()
cols_geracao = ['geracao_eolica_mwmed', 'geracao_solar_mwmed', 'geracao_hidreletrica_mwmed', 'geracao_termica_mwmed']
df[cols_geracao] = df[cols_geracao].interpolate(method='linear')

# --- MODELO 1: Alvo = Geração Solar ---
X_solar = df[['preco_brent', 'geracao_hidreletrica_mwmed', 'geracao_termica_mwmed', 'geracao_eolica_mwmed']]
y_solar = df['geracao_solar_mwmed']

rf_solar = RandomForestRegressor(n_estimators=1000, random_state=42)
rf_solar.fit(X_solar, y_solar)

importances_solar = pd.Series(rf_solar.feature_importances_ * 100, index=X_solar.columns)

# --- MODELO 2: Alvo = Geração Eólica ---
X_eolica = df[['preco_brent', 'geracao_hidreletrica_mwmed', 'geracao_termica_mwmed', 'geracao_solar_mwmed']]
y_eolica = df['geracao_eolica_mwmed']

rf_eolica = RandomForestRegressor(n_estimators=100, random_state=42)
rf_eolica.fit(X_eolica, y_eolica)

importances_eolica = pd.Series(rf_eolica.feature_importances_ * 100, index=X_eolica.columns)

# Consolidação dos resultados
df_importances = pd.DataFrame({
    'Peso para Solar (%)': importances_solar,
    'Peso para Eólica (%)': importances_eolica
}).fillna(0)

print(df_importances.round(2))

                            Peso para Solar (%)  Peso para Eólica (%)
geracao_eolica_mwmed                      56.37                  0.00
geracao_hidreletrica_mwmed                14.91                 13.11
geracao_solar_mwmed                        0.00                 81.21
geracao_termica_mwmed                     11.10                  2.52
preco_brent                               17.62                  3.16
